Logistics Shipments Dataset - Data cleaning

This notebook focuses on cleaning and preparing the logistics shipment dataset
for analytical modeling.

The exploratory analysis identified several data quality issues including
incorrect data types, missing values, and textual entries in numeric fields.
These issues are addressed in this stage to ensure consistent analysis.

Data Quality Issues Identified

The exploratory analysis revealed the following issues:

- Date columns stored as text
- Numeric columns stored as strings
- Textual references in numeric fields (e.g. "See ASN-93")
- Missing values in some columns

These issues will be addressed in the following steps.

In [15]:
import pandas as pd

df = pd.read_csv("../data/raw/shipments_raw.csv")

df.shape

#The dataset contains 10,324 shipment records.

(10324, 33)

In [16]:
df["PQ First Sent to Client Date"].unique()

<StringArray>
[   'Pre-PQ Process', 'Date Not Captured',          '11/18/09',
            '5/3/13',           '8/19/14',            '1/6/12',
           '2/22/13',          '10/28/14',           '2/20/13',
           '2/17/12',
 ...
           '11/1/11',           '1/16/12',           '3/16/13',
           '2/26/15',           '2/12/10',           '12/3/13',
          '12/19/14',            '8/2/11',            '2/2/12',
           '8/29/13']
Length: 765, dtype: str

In [17]:
df["PO Sent to Vendor Date"].unique()

<StringArray>
['Date Not Captured',          '11/13/06',           '12/1/06',
          '12/22/06',           '1/10/07',           '4/12/07',
           '5/13/07',           '5/17/07',           '7/13/07',
            '7/4/07',
 ...
           '1/31/12',            '6/3/14',           '8/11/10',
           '5/26/11',          '10/13/11',           '2/20/13',
           '9/26/12',           '12/3/13',            '3/9/10',
           '8/29/12']
Length: 897, dtype: str

In [ ]:
df["pq_sent_to_client_date"] = pd.to_datetime(
    df["PQ First Sent to Client Date"],
    errors="coerce"
)

df["po_sent_to_vendor_date"] = pd.to_datetime(
    df["PO Sent to Vendor Date"],
    errors="coerce"
)

C:\Users\danie\AppData\Local\Temp\ipykernel_9152\3375178345.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["pq_sent_to_client_date"] = pd.to_datetime(
C:\Users\danie\AppData\Local\Temp\ipykernel_9152\3375178345.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["po_sent_to_vendor_date"] = pd.to_datetime(


In [19]:
df["pq_sent_to_client_status"] = df["PQ First Sent to Client Date"].where(
    df["pq_sent_to_client_date"].isna(),
    None
)

df["po_sent_to_vendor_status"] = df["PO Sent to Vendor Date"].where(
    df["po_sent_to_vendor_date"].isna(),
    None
)

In [20]:
df["pq_sent_to_client_status"] = df["pq_sent_to_client_status"].replace({
    "Date Not Captured": "Missing",
})

df["po_sent_to_vendor_status"] = df["po_sent_to_vendor_status"].replace({
    "Date Not Captured": "Missing",
    "N/A - From RDC": "Not Applicable - From RDC"
})

In [21]:
df["Scheduled Delivery Date"] = pd.to_datetime(
    df["Scheduled Delivery Date"],
    errors="coerce"
)

df["Delivered to Client Date"] = pd.to_datetime(
    df["Delivered to Client Date"],
    errors="coerce"
)

df["Delivery Recorded Date"] = pd.to_datetime(
    df["Delivery Recorded Date"],
    errors="coerce"
)

C:\Users\danie\AppData\Local\Temp\ipykernel_9152\3769262772.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Scheduled Delivery Date"] = pd.to_datetime(
C:\Users\danie\AppData\Local\Temp\ipykernel_9152\3769262772.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Delivered to Client Date"] = pd.to_datetime(
C:\Users\danie\AppData\Local\Temp\ipykernel_9152\3769262772.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Delivery Recorded Date"] = pd.to_datetime(


In [25]:
# Validation of date columns after conversion
df[
    ["pq_sent_to_client_date", "pq_sent_to_client_status", "po_sent_to_vendor_date", 
     "po_sent_to_vendor_status", "Scheduled Delivery Date", "Delivered to Client Date", 
     "Delivery Recorded Date"]
].info()

<class 'pandas.DataFrame'>
RangeIndex: 10324 entries, 0 to 10323
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   pq_sent_to_client_date    7643 non-null   datetime64[us]
 1   pq_sent_to_client_status  2681 non-null   str           
 2   po_sent_to_vendor_date    4592 non-null   datetime64[us]
 3   po_sent_to_vendor_status  5732 non-null   str           
 4   Scheduled Delivery Date   10324 non-null  datetime64[us]
 5   Delivered to Client Date  10324 non-null  datetime64[us]
 6   Delivery Recorded Date    10324 non-null  datetime64[us]
dtypes: datetime64[us](5), str(2)
memory usage: 564.7 KB


Splitting Date, Process Status and Date Format

The columns "PQ First Sent to Client Date" and "PO Sent to Vendor Date" contained both valid dates and textual process indicators such as "Pre-PQ Process", "Date Not Captured" and "N/A - From RDC".

To preserve this information:

- New datetime columns (pq_sent_to_client_date, po_sent_to_vendor_date) were created
- Separate categorical columns (pq_sent_to_client_status, po_sent_to_vendor_status) were created to retain non-date values

This allows distinguishing between actual timestamps and process states.

The columns "Scheduled Delivery Date", "Delivered to Client Date", "Delivery Recorded Date" stored as strings were converted to datetime format to enable temporal analysis.

In [ ]:
# Convert weight and freight cost to numeric, coercing errors to NaN
df["weight_kg"] = pd.to_numeric(
    df["Weight (Kilograms)"],
    errors="coerce"
)
df["freight_cost_usd"] = pd.to_numeric(
    df["Freight Cost (USD)"],
    errors="coerce"
)

# Create details columns for weight and freight cost where original values are preserved if conversion failed
df["weight_kg_details"] = df["Weight (Kilograms)"].where(
    df["weight_kg"].isna(),
    None
)
df["freight_cost_usd_details"] = df["Freight Cost (USD)"].where(
    df["freight_cost_usd"].isna(),
    None
)

df[
    ["weight_kg", "freight_cost_usd", "weight_kg_details", "freight_cost_usd_details"]
].info()

<class 'pandas.DataFrame'>
RangeIndex: 10324 entries, 0 to 10323
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   weight_kg                 6372 non-null   float64
 1   freight_cost_usd          6198 non-null   float64
 2   weight_kg_details         3952 non-null   str    
 3   freight_cost_usd_details  4126 non-null   str    
dtypes: float64(2), str(2)
memory usage: 322.8 KB


Cleaning Numeric Columns

Some numeric columns contained non-numeric text values such as:

- "See ASN-93 (ID#:1281)"
- "Weight Captured Separately"
- "Freight Included in Commodity Cost"
- "Invoiced Separately"

These entries indicate that the corresponding values were either recorded
in a different record or handled outside the dataset.

For analytical purposes:

- The columns "Weight (Kilograms)" and "Freight Cost (USD)" were converted
  to numeric format, coercing invalid entries to NaN.

- Original non-numeric values were preserved in separate columns:
  "weight_kg_details" and "freight_cost_usd_details".

This allows identifying relationships between records without introducing
assumptions by automatically imputing values.

In [42]:
df[["ID",
     "Weight (Kilograms)", "weight_kg", "weight_kg_details",
     "Freight Cost (USD)", "freight_cost_usd", "freight_cost_usd_details"]].tail(10)

,ID,Weight (Kilograms),weight_kg,weight_kg_details,Freight Cost (USD),freight_cost_usd,freight_cost_usd_details
10314,86813,See DN-4274 (ID#:84472),NaN,See DN-4274 (ID#:84472),See DN-4274 (ID#:84472),NaN,See DN-4274 (ID#:84472)
10315,86814,15198,15198.0,NaN,26180,26180.0,NaN
10316,86815,1547,1547.0,NaN,3410,3410.0,NaN
10317,86816,See DN-4282 (ID#:83919),NaN,See DN-4282 (ID#:83919),See DN-4282 (ID#:83919),NaN,See DN-4282 (ID#:83919)
10318,86817,See DN-4307 (ID#:83920),NaN,See DN-4307 (ID#:83920),See DN-4307 (ID#:83920),NaN,See DN-4307 (ID#:83920)
10319,86818,See DN-4307 (ID#:83920),NaN,See DN-4307 (ID#:83920),See DN-4307 (ID#:83920),NaN,See DN-4307 (ID#:83920)
10320,86819,See DN-4313 (ID#:83921),NaN,See DN-4313 (ID#:83921),See DN-4313 (ID#:83921),NaN,See DN-4313 (ID#:83921)
10321,86821,Weight Captured Separately,NaN,Weight Captured Separately,Freight Included in Commodity Cost,NaN,Freight Included in Commodity Cost
10322,86822,1392,1392.0,NaN,Freight Included in Commodity Cost,NaN,Freight Included in Commodity Cost
10323,86823,Weight Captured Separately,NaN,Weight Captured Separately,Freight Included in Commodity Cost,NaN,Freight Included in Commodity Cost


In [45]:
import numpy as np

# Calculate cost per kg, handling division by zero and missing values
df["cost_per_kg"] = df["freight_cost_usd"] / df["weight_kg"].replace(0, np.nan)


df["delivery_delay_days"] = (
    df["Delivered to Client Date"] -
    df["Scheduled Delivery Date"]
).dt.days

In [50]:
df[["ID",
     "Delivered to Client Date", "Scheduled Delivery Date", "delivery_delay_days", 
     "freight_cost_usd", "weight_kg", "cost_per_kg"]].iloc[20:25]

,ID,Delivered to Client Date,Scheduled Delivery Date,delivery_delay_days,freight_cost_usd,weight_kg,cost_per_kg
20,96,2007-06-19,2007-06-19,0,15893.71,2278.0,6.977046
21,108,2007-10-02,2007-10-02,0,NaN,2126.0,NaN
22,115,2007-10-15,2007-10-15,0,4193.49,941.0,4.456419
23,116,2007-08-27,2007-08-27,0,1767.38,117.0,15.105812
24,130,2007-08-21,2007-08-13,8,3518.38,171.0,20.575322


Creating Analytical Features

Additional variables were created to support logistics analysis ("cost_per_kg", "delivery_delay_days").

In [51]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10324 entries, 0 to 10323
Data columns (total 43 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   ID                            10324 non-null  int64         
 1   Project Code                  10324 non-null  str           
 2   PQ #                          10324 non-null  str           
 3   PO / SO #                     10324 non-null  str           
 4   ASN/DN #                      10324 non-null  str           
 5   Country                       10324 non-null  str           
 6   Managed By                    10324 non-null  str           
 7   Fulfill Via                   10324 non-null  str           
 8   Vendor INCO Term              10324 non-null  str           
 9   Shipment Mode                 9964 non-null   str           
 10  PQ First Sent to Client Date  10324 non-null  str           
 11  PO Sent to Vendor Date        10324 non

Final Dataset Validation

After cleaning and transformations, the dataset was validated to ensure correct data types and consistency across key analytical fields.

In [52]:
df.to_csv("../data/processed/shipments_clean.csv", index=False)

Exporting Clean Dataset

The cleaned dataset is exported for further analysis and modeling.